# Phase 2: Quantum SVM benchmark on parity dataset

This notebook benchmarks a Quantum SVM (QSVC) on parity datasets using Qiskit Machine Learning.
We compare results against classical SVM baselines to explore potential quantum advantage.

Import Qiskit’s QSVC, scikit‑learn utilities, and datetime for timestamping.

In [1]:
import sys, os, time
import numpy as np
from qiskit_machine_learning.algorithms import QSVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from datetime import datetime, timezone

Ensure notebook can import custom utilities from src/utils.

In [2]:
import os, sys

# Notebook is in /app/src/phase2 → go up one level to /app/src
SRC_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

Custom utilities for logging results, plotting decision boundaries, and loading datasets.

In [3]:
from utils.logger import log_results
from utils.visualizer import plot_projected_decision_boundary
from utils.data_loader import load_dataset_from_config

Load dataset via config, separate features (X) and target (y).

In [4]:
df, cfg = load_dataset_from_config()

X = df.drop(columns=["target"]).values
y = df["target"].values

Use a fixed 50/50 split for parity experiments.

In [5]:
# Fixed 50/50 split for parity experiments
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

 Instantiate QSVC — backend and shots are not configurable in this version.

In [6]:
# QSVC builds its own kernel internally (v0.8.4)
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.algorithms import QSVC

# Define a feature map
feature_map = ZZFeatureMap(feature_dimension=X.shape[1])

# Pass it into QSVC
qsvc = QSVC(feature_map=feature_map)

/tmp/ipykernel_1809/1378936259.py:9: QiskitMachineLearningWarning: 'No quantum kernel is provided, SamplerV1 based quantum kernel will be used.'
  qsvc = QSVC(feature_map=feature_map)


TypeError: SVC.__init__() got an unexpected keyword argument 'feature_map'

Train QSVM on parity dataset.

In [ ]:
start = time.time()
qsvc.fit(X_train, y_train)
training_time = round(time.time() - start, 4)

Evaluate accuracy, generalization gap, log results to CSV, and print metrics.

In [ ]:
y_train_pred = qsvc.predict(X_train)
y_test_pred = qsvc.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
generalization_gap = round(train_accuracy - test_accuracy, 4)

metrics = {
    "model": "QSVM_Parity",
    "dataset": cfg["dataset"],
    "accuracy": test_accuracy,
    "train_accuracy": train_accuracy,
    "generalization_gap": generalization_gap,
    "training_time": training_time
}
log_results(metrics)

print("\n=== QSVM Parity Results ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

Visualize QSVM decision boundary in PCA‑projected space and save plot with timestamped filename.

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
plot_filename = f"{metrics['model'].lower()}_{metrics['dataset']}_pca_projection_{timestamp}.png"

plot_projected_decision_boundary(
    qsvc,
    X_test,
    y_test,
    title="QSVM Parity (PCA Projection)",
    filename=plot_filename
)